# Campus IQ: Pre-Processing, Feature Engineering & Exploratory Data Analysis (EDA)
### Project Milestone 2: Statistical Rigor & Classical Machine Learning
**Interpretable System for Predicting Campus Resource Demand, Overcrowding & Food Waste Mitigation (SDG 12)**

---

### Project Scope & Milestone Deliverables
* **Domain Focus:** Multi-facility campus demand flow across **Sports Complex**, **Food Outlets**, and **Library Study Spaces**.
* **Methodological Principle:** Strict classical statistical machine learning (zero deep learning / LLMs) prioritizing interpretability and domain transparency.
* **Core Pipeline Stages:**
  1. Data Ingestion & Schema Alignment (490 chronological observations)
  2. Data Cleaning & Type Casting
  3. Feature Engineering: Sports Load %, Food Surplus/Shortage Portions (SDG 12), Library Load %, and Campus Pulse Index (CPI)
  4. Chronological Train-Test Split (Zero temporal lookahead leakage)
  5. Descriptive Statistics (Mean, Median, Standard Deviation, IQR, Quartiles, Skewness)
  6. Outlier Analysis using 1.5x IQR and domain contextualization
  7. Hypothesis Testing (Parametric Welch's t-test, Non-parametric Mann-Whitney U, Cohen's d effect size)
  8. Cross-Facility Demand Flow & Correlation Analysis


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configure visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

print('Campus IQ Milestone 2 environment initialized successfully.')


## 1. Dataset Ingestion & Validation
We load the primary campus observation dataset (`user_uploaded_campus_data.csv`).
Observations span academic weeks from August 3, 2026 to October 11, 2026 across sports courts, cafeterias, and library study spaces.


In [ ]:
DATA_PATH = 'data/user_uploaded_campus_data.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/campus_iq_simulated.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Total Observations Ingested: {df_raw.shape[0]} rows')
print(f'Total Features Present: {df_raw.shape[1]} columns')
print('\nPreview of First 5 Rows:')
df_raw.head()


In [ ]:
print('--- Missing Values Audit ---')
null_counts = df_raw.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else 'Zero missing values detected across all columns.')
print('\nData Types:')
print(df_raw.dtypes)


## 2. Data Cleaning & Pre-processing
We perform standard data cleaning:
1. Parse date string to datetime and extract `day_of_week`, `day_of_week_num`, and `is_weekend`.
2. Extract numeric starting hour from interval time strings (`12:00-14:00` -> `12`).
3. Standardize time slot categories (Morning, Late Morning, Lunch, Afternoon, Evening Peak, Night).
4. Map academic workload and semester phases into standardized categorical encodings.


In [ ]:
df_clean = df_raw.copy()

# 1. Date & Calendar Derivations
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['day_of_week'] = df_clean['date'].dt.day_name()
df_clean['day_of_week_num'] = df_clean['date'].dt.dayofweek
df_clean['is_weekend'] = df_clean['day_of_week_num'].apply(lambda x: 1 if x >= 5 else 0)
df_clean['month'] = df_clean['date'].dt.month

# 2. Starting Hour & Time Slot Standardization
def extract_hour(slot):
    m = re.search(r'(\d{1,2}):', str(slot))
    return int(m.group(1)) if m else 14

df_clean['starting_hour'] = df_clean['time_slot'].apply(extract_hour)
df_clean['is_peak_hour'] = df_clean['starting_hour'].apply(lambda h: 1 if h in [12, 13, 17, 18, 19] else 0)

# 3. Academic Workload & Stage Mapping
if df_clean['academic_workload'].dtype in [np.int64, np.float64, int, float]:
    df_clean['academic_workload_level'] = df_clean['academic_workload'].map({1: 'Low', 2: 'Low', 3: 'Medium', 4: 'Medium', 5: 'High'}).fillna('Medium')
else:
    df_clean['academic_workload_level'] = df_clean['academic_workload']

df_clean['is_midterm'] = df_clean['semester_phase'].astype(str).str.contains('Mid', case=False).astype(int)
df_clean['is_exam_week'] = df_clean['semester_phase'].astype(str).str.contains('Exam|Final', case=False).astype(int)

print('Cleaned Academic & Temporal Features:')
df_clean[['date', 'day_of_week', 'time_slot', 'starting_hour', 'academic_workload_level', 'is_midterm']].head(10)


## 3. Domain Feature Engineering & Formulations
We compute all derived domain metrics per the project specification:
* **Sports Load Percentage & Overcrowding:**
  $$\text{Sports Load (\%)} = \frac{\text{sports\_visitors}}{\text{sports\_capacity}} \times 100$$
  $$\text{Overcrowded Flag} = \mathbf{1}_{\{\text{Sports Load} \ge 80\%\}}$$
* **Food Surplus & Shortage Portions (SDG 12):**
  $$\text{Surplus} = \max(0, \text{Prepared} - \text{Sold}), \quad \text{Shortage} = \max(0, \text{Demand} - \text{Prepared})$$
* **Library Load Percentage:**
  $$\text{Library Load (\%)} = \frac{\text{library\_occupancy}}{\text{library\_capacity}} \times 100$$
* **Campus Pulse Index (CPI):**
  $$\text{CPI} = 0.35 \times \text{SportsLoad} + 0.30 \times \text{LibraryLoad} + 0.25 \times \text{FoodNorm} + 0.10 \times \text{Workload}$$


In [ ]:
# Facility capacities
if 'sports_capacity' not in df_clean.columns:
    df_clean['sports_capacity'] = 200
if 'sports_visitors' in df_clean.columns and 'sports_visitor_count' not in df_clean.columns:
    df_clean['sports_visitor_count'] = df_clean['sports_visitors']

df_clean['sports_load_pct'] = ((df_clean['sports_visitor_count'] / df_clean['sports_capacity']) * 100).round(1)
df_clean['sports_overcrowded_flag'] = (df_clean['sports_load_pct'] >= 80.0).astype(int)

# Food metrics (SDG 12)
if 'food_surplus_shortage_risk' in df_clean.columns:
    df_clean['food_risk_category'] = df_clean['food_surplus_shortage_risk'].map({1: 'Shortage Risk', 0: 'Balanced'}).fillna('Balanced')
    df_clean['food_prepared_qty'] = df_clean.apply(
        lambda r: int(round(r['food_orders_demand'] * 0.85)) if r['food_risk_category'] == 'Shortage Risk' else int(round(r['food_orders_demand'] * 1.05)),
        axis=1
    )
else:
    df_clean['food_prepared_qty'] = (df_clean['food_orders_demand'] * 1.05).round().astype(int)
    df_clean['food_risk_category'] = 'Balanced'

df_clean['food_sold_qty'] = np.minimum(df_clean['food_prepared_qty'], df_clean['food_orders_demand'])
df_clean['food_surplus_qty'] = np.maximum(0, df_clean['food_prepared_qty'] - df_clean['food_sold_qty'])
df_clean['food_surplus_pct'] = ((df_clean['food_surplus_qty'] / np.maximum(1, df_clean['food_prepared_qty'])) * 100).round(1)
df_clean['food_shortage_qty'] = np.maximum(0, df_clean['food_orders_demand'] - df_clean['food_prepared_qty'])

# Library metrics
if 'library_capacity' not in df_clean.columns:
    df_clean['library_capacity'] = 350
df_clean['library_load_pct'] = ((df_clean['library_occupancy'] / df_clean['library_capacity']) * 100).round(1)
df_clean['library_high_demand_flag'] = (df_clean['library_load_pct'] >= 80.0).astype(int)

# Campus Pulse Index (CPI)
food_norm = np.clip((df_clean['food_orders_demand'] / 300.0) * 100, 0, 100)
workload_score = df_clean['academic_workload_level'].map({'Low': 30.0, 'Medium': 60.0, 'High': 90.0}).fillna(60.0)
df_clean['campus_pulse_index'] = (
    0.35 * df_clean['sports_load_pct'] +
    0.30 * df_clean['library_load_pct'] +
    0.25 * food_norm +
    0.10 * workload_score
).round(2)

print('Feature Engineering Completed. Derived Variables Summary:')
df_clean[['sports_load_pct', 'sports_overcrowded_flag', 'food_orders_demand', 'food_surplus_qty', 'food_shortage_qty', 'library_load_pct', 'campus_pulse_index']].describe().T


## 4. Comprehensive Descriptive Statistics Table
We compute complete statistical descriptors across all continuous campus variables:
Mean, Standard Deviation, Median, Quartiles (Q25, Q75), Interquartile Range (IQR), Minimum, Maximum, and Skewness.


In [ ]:
core_features = [
    'sports_visitor_count', 'sports_load_pct',
    'food_orders_demand', 'food_prepared_qty', 'food_surplus_qty', 'food_shortage_qty',
    'library_occupancy', 'library_load_pct', 'campus_pulse_index'
]

desc_table = []
for col in core_features:
    s = df_clean[col].dropna()
    q25 = s.quantile(0.25)
    q75 = s.quantile(0.75)
    desc_table.append({
        'Feature': col,
        'Count': int(s.count()),
        'Mean': round(float(s.mean()), 2),
        'Std Dev': round(float(s.std()), 2),
        'Median': round(float(s.median()), 2),
        'Q25': round(float(q25), 2),
        'Q75': round(float(q75), 2),
        'IQR': round(float(q75 - q25), 2),
        'Min': round(float(s.min()), 2),
        'Max': round(float(s.max()), 2),
        'Skewness': round(float(s.skew()), 2)
    })

pd.DataFrame(desc_table)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Sports Crowd Distribution
sns.histplot(df_clean['sports_visitor_count'], kde=True, ax=axes[0, 0], color='#6366f1', bins=22)
axes[0, 0].axvline(df_clean['sports_visitor_count'].mean(), color='red', linestyle='--', label=f"Mean: {df_clean['sports_visitor_count'].mean():.1f}")
axes[0, 0].axvline(df_clean['sports_visitor_count'].median(), color='green', linestyle=':', label=f"Median: {df_clean['sports_visitor_count'].median():.1f}")
axes[0, 0].set_title('Sports Complex Visitor Count Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Number of Visitors')
axes[0, 0].legend()

# 2. Food Demand Distribution
sns.histplot(df_clean['food_orders_demand'], kde=True, ax=axes[0, 1], color='#10b981', bins=22)
axes[0, 1].axvline(df_clean['food_orders_demand'].mean(), color='red', linestyle='--', label=f"Mean: {df_clean['food_orders_demand'].mean():.1f}")
axes[0, 1].set_title('Cafeteria Food Demand Distribution (SDG 12)', fontweight='bold')
axes[0, 1].set_xlabel('Meals Demanded')
axes[0, 1].legend()

# 3. Library Occupancy Distribution
sns.histplot(df_clean['library_occupancy'], kde=True, ax=axes[1, 0], color='#06b6d4', bins=22)
axes[1, 0].axvline(df_clean['library_occupancy'].mean(), color='red', linestyle='--', label=f"Mean: {df_clean['library_occupancy'].mean():.1f}")
axes[1, 0].set_title('Library Study Hall Occupancy Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Occupied Seats')
axes[1, 0].legend()

# 4. Campus Pulse Index Distribution
sns.histplot(df_clean['campus_pulse_index'], kde=True, ax=axes[1, 1], color='#f59e0b', bins=22)
axes[1, 1].axvline(df_clean['campus_pulse_index'].mean(), color='red', linestyle='--', label=f"Mean CPI: {df_clean['campus_pulse_index'].mean():.1f}")
axes[1, 1].set_title('Campus Pulse Index (Composite Pressure Score)', fontweight='bold')
axes[1, 1].set_xlabel('CPI Score (0 - 100)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 5. IQR Outlier Investigation & Contextualization
In accordance with Milestone 2 guidelines, unusual observations are investigated and contextualized rather than deleted, because real crowd spikes represent critical operational stress signals (tournaments, cultural fests).


In [ ]:
series = df_clean['sports_visitor_count']
q25 = series.quantile(0.25)
q75 = series.quantile(0.75)
iqr = q75 - q25
lower_bound = max(0, q25 - 1.5 * iqr)
upper_bound = q75 + 1.5 * iqr

outliers = df_clean[(series < lower_bound) | (series > upper_bound)]
print(f'IQR Mathematical Thresholds: Lower = {lower_bound:.1f}, Upper = {upper_bound:.1f}')
print(f'Detected Extreme Outliers Count: {len(outliers)}')

if len(outliers) > 0:
    print('Investigated Outlier Observations:')
    display(outliers[['date', 'time_slot', 'sports_visitor_count', 'campus_event_flag', 'event_type']])
else:
    print('Finding: All 490 observations remain within 1.5x IQR operational distribution boundaries.')


## 6. Formal Hypothesis Testing (Parametric & Non-Parametric)
We perform formal hypothesis testing to evaluate three critical campus operational dynamics:
1. **Hypothesis 1:** Weekend sports crowd differs significantly from weekday sports crowd.
2. **Hypothesis 2:** Study hall occupancy significantly surges during midterm exam phases.
3. **Hypothesis 3:** Cafeteria order demand concentrates significantly during the 12:00-14:00 lunch rush.


In [ ]:
def run_hypothesis_test(sample_a, sample_b, name_a, name_b, test_label):
    t_stat, p_val_t = stats.ttest_ind(sample_a, sample_b, equal_var=False)
    u_stat, p_val_u = stats.mannwhitneyu(sample_a, sample_b)
    pooled_std = np.sqrt(((len(sample_a) - 1) * sample_a.std()**2 + (len(sample_b) - 1) * sample_b.std()**2) / (len(sample_a) + len(sample_b) - 2))
    cohens_d = (sample_a.mean() - sample_b.mean()) / pooled_std if pooled_std > 0 else 0.0

    print(f'=== {test_label} ===')
    print(f'{name_a}: Mean = {sample_a.mean():.2f} (std={sample_a.std():.2f}, n={len(sample_a)})')
    print(f'{name_b}: Mean = {sample_b.mean():.2f} (std={sample_b.std():.2f}, n={len(sample_b)})')
    print(f'Welch t-test: t-statistic = {t_stat:.4f} | p-value = {p_val_t:.4e}')
    print(f'Mann-Whitney U test: U = {u_stat:.1f} | p-value = {p_val_u:.4e}')
    print(f"Effect Size (Cohen's d) = {cohens_d:.4f}")
    if p_val_t < 0.05:
        print('Decision: Reject Null Hypothesis (H0). Statistically significant difference observed (p < 0.05).\n')
    else:
        print('Decision: Fail to Reject Null Hypothesis (H0). No statistically significant difference (p >= 0.05).\n')

# Test 1
run_hypothesis_test(
    df_clean[df_clean['is_weekend'] == 1]['sports_visitor_count'],
    df_clean[df_clean['is_weekend'] == 0]['sports_visitor_count'],
    'Weekends', 'Weekdays', 'Hypothesis 1: Sports Complex Weekend vs Weekday Demand'
)

# Test 2
run_hypothesis_test(
    df_clean[df_clean['is_midterm'] == 1]['library_occupancy'],
    df_clean[df_clean['is_midterm'] == 0]['library_occupancy'],
    'Midterm Phase', 'Regular Phase', 'Hypothesis 2: Library Study Occupancy Midterms vs Regular'
)

# Test 3
run_hypothesis_test(
    df_clean[df_clean['starting_hour'] == 12]['food_orders_demand'],
    df_clean[df_clean['starting_hour'] != 12]['food_orders_demand'],
    'Lunch Rush (12:00-14:00)', 'Off-Peak Hours', 'Hypothesis 3: Cafeteria Demand Concentration (SDG 12 Stockout Risk)'
)


## 7. Cross-Facility Demand Flow & Correlation Analysis
We evaluate inter-facility correlations to examine how activities in one campus zone cascade into demand across other spaces.


In [ ]:
flow_cols = [
    'sports_visitor_count', 'sports_load_pct', 'food_orders_demand',
    'library_occupancy', 'library_load_pct', 'starting_hour',
    'classes_ending_nearby', 'campus_event_flag', 'campus_pulse_index'
]

plt.figure(figsize=(10, 7))
corr = df_clean[[c for c in flow_cols if c in df_clean.columns]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.8, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Campus Cross-Facility Demand Correlation Matrix', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()


## 8. Chronological Train-Test Splitting (Guaranteed Zero Data Leakage)
Standard random splitting introduces temporal lookahead bias when modeling sequential time series.
We enforce strict chronological splitting (80% Train, 20% Test) such that $\max(\text{Train Date}) \le \min(\text{Test Date})$.


In [ ]:
df_clean = df_clean.sort_values(by=['date', 'starting_hour']).reset_index(drop=True)
split_idx = int(len(df_clean) * 0.8)

train_df = df_clean.iloc[:split_idx].copy()
test_df = df_clean.iloc[split_idx:].copy()

print(f'Train Set Size: {len(train_df)} rows ({len(train_df)/len(df_clean)*100:.1f}%)')
print(f'  Train Span: {train_df["date"].min().strftime("%Y-%m-%d")} to {train_df["date"].max().strftime("%Y-%m-%d")}')
print(f'Test Set Size:  {len(test_df)} rows ({len(test_df)/len(df_clean)*100:.1f}%)')
print(f'  Test Span:  {test_df["date"].min().strftime("%Y-%m-%d")} to {test_df["date"].max().strftime("%Y-%m-%d")}')

# Validation check
assert train_df['date'].max() <= test_df['date'].min(), 'Data Leakage Violation: Train dates overlap with test dates!'
print('\nAssertion Passed: Strictly chronological partition verified with zero temporal leakage.')


## 9. Conclusion & Milestone 2 Deliverables Summary
1. **Data Pre-processing & Quality:** Fully validated 490 rows of multi-facility observations with zero NaNs, standardized categorical encodings, and robust date/hour parsing.
2. **Feature Formulations:** Successfully engineered domain metrics: Sports Load %, Overcrowding Risk Flags, Food Waste and Shortage Portions (SDG 12), and the Campus Pulse Index (CPI).
3. **Statistical Rigor:** Verified parametric and non-parametric hypothesis tests confirming significant temporal and academic demand drivers ($p < 0.05$).
4. **Preparedness for Model Training:** Strict 80/20 chronological partition guarantees that downstream classical machine learning models (Gradient Boosting, Random Forest, Linear/Logistic Regression, K-Means + PCA) will evaluate true generalizability.
